---
# 🚀 **Project:** Fine Tuned Java Vulnerability-Aware Qwen3-Coder-Next Model
**Hackathon Challenge:** Track 3 – Fine Tuning (Advanced Users)

**Use Case Details:** FINETUNING_001: Vulnerability-Aware Java Model.

* **Base Model:** `Qwen/Qwen3-Coder-Next` (80B MoE)
* **Optimization:** LoRA (Rank-64) via Unsloth PEFT
* **Hardware:** AMD MI300X (192GB VRAM)
---

# 🛠️ Phase 1: Environment Setup & Self-Healing Dependencies
**Objective:** Prepare the runtime environment and inject AMD ROCm-compatible acceleration kernels.
* Installs and verifies `unsloth` for optimized training.
* Implements self-healing logic for `flash-linear-attention` and `causal-conv1d`.
* Verifies Gated DeltaNet fast kernels to ensure high-speed generation on AMD hardware.

In [1]:
# ---- Cell 1: Unsloth FIRST, then self-healing deps ----
import os, subprocess
os.environ["UNSLOTH_SKIP_TORCHVISION_CHECK"] = "1"

import unsloth
from unsloth import FastLanguageModel
print(f"✅ Unsloth {unsloth.__version__} | FastLanguageModel ready")

# ── Eval libs (silent) ──
subprocess.run("pip install -q --root-user-action=ignore rouge_score evaluate sacrebleu nltk".split(),
               capture_output=True)
print("✅ Eval libs installed")

# ── Self-healing: flash-linear-attention ──
try:
    import fla
    print(f"✅ flash-linear-attention {fla.__version__} ready")
except ImportError:
    print("📦 Installing flash-linear-attention...")
    subprocess.run("pip install --root-user-action=ignore flash-linear-attention".split())
    import fla
    print(f"✅ flash-linear-attention {fla.__version__} installed & ready")

# ── Self-healing: causal-conv1d (needs --no-build-isolation for ROCm) ──
try:
    from causal_conv1d import causal_conv1d_fn
    print("✅ causal-conv1d ready")
except ImportError:
    print("📦 Installing causal-conv1d (compiling for ROCm — ~5-15 min)...")
    ret = subprocess.run(
        "pip install --root-user-action=ignore --no-build-isolation causal-conv1d".split()
    )
    if ret.returncode == 0:
        from causal_conv1d import causal_conv1d_fn
        print("✅ causal-conv1d installed & ready")
    else:
        print("⚠️ causal-conv1d failed to build — fla alone should still enable fast path")

# ── Final verification: the kernel that actually matters ──
try:
    from fla.ops.gated_delta_rule import fused_recurrent_gated_delta_rule
    print("✅ Gated DeltaNet fast kernels verified — generation will be FAST")
except ImportError:
    print("❌ WARNING: Gated DeltaNet fast kernels NOT available — generation will be SLOW")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/unsloth_zoo/temporary_patches/misc.py:959: UserWarning: expandable_segments not supported on this platform (Triggered internally at /app/pytorch/c10/hip/HIPAllocatorConfig.h:36.)
  x = torch.randn(1, 4, 8, device=device, dtype=torch.float16)


INFO 06-15 18:43:38 [__init__.py:225] Automatically detected platform rocm.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ Unsloth 2026.6.7 | FastLanguageModel ready
✅ Eval libs installed
✅ flash-linear-attention 0.5.0 ready
✅ causal-conv1d ready
✅ Gated DeltaNet fast kernels verified — generation will be FAST


# 🖥️ Phase 2: Hardware Verification & Pipeline Configuration
**Objective:** Validate the compute environment and define global training parameters.
* Executes `rocm-smi` to confirm **AMD MI300X** allocation.
* Verifies PyTorch/HIP integration and ensures >50GB of free disk space for LoRA checkpoints.
* Optimizes VRAM allocation by enabling `expandable_segments` to prevent fragmentation.
* Locks `LOAD_IN_4BIT = False` to maintain BF16 precision and bypass ROCm quantization instability.

In [2]:
# ---- Cell 2: Hardware & Config ----

import torch, os, json, re, gc, shutil, subprocess

from collections import Counter

print("=" * 55)

print("🖥️  HARDWARE REPORT")

print("=" * 55)

# ROCm system info

result = subprocess.run(["rocm-smi", "--showproductname"], capture_output=True, text=True)

if result.returncode == 0:

    for line in result.stdout.strip().splitlines():

        if line.strip():

            print(f"  {line.strip()}")

print(f"  PyTorch  : {torch.__version__}")

print(f"  HIP      : {torch.version.hip}")

print(f"  GPU avail: {torch.cuda.is_available()}")

for i in range(torch.cuda.device_count()):

    p = torch.cuda.get_device_properties(i)

    print(f"  GPU {i}    : {p.name}  |  {p.total_memory/1e9:.1f} GB")

total, used, free = shutil.disk_usage("/")

print(f"  Disk free: {free/1e9:.0f} GB")

if free > 50e9:

    print("  ✅ Enough disk for LoRA adapter training")

else:

    raise RuntimeError("❌ Not enough disk")

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# ── Config block ──

MODEL_PATH   = "./models/Qwen3-Coder-Next"

ADAPTER_DIR  = "./vuln_qwen_adapter"

FINAL_DIR    = "./vuln_qwen_adapter"     # no bf16 merge — serve adapter directly

OUTPUT_DIR   = "./training_output"

MAX_SEQ_LEN  = 2048

LOAD_IN_4BIT = False                     # bf16 on MI300X — 4bit unstable on ROCm

print(f"\n  Model    : {MODEL_PATH}")

print(f"  Adapter  : {ADAPTER_DIR}")

print(f"  4-bit    : {LOAD_IN_4BIT}")

print(f"  Max seq  : {MAX_SEQ_LEN}")

🖥️  HARDWARE REPORT
  ============================ ROCm System Management Interface ============================
  ====================================== Product Info ======================================
  GPU[0]		: get_name, Error when calling libdrm
  GPU[0]		: Card Series: 		N/A
  GPU[0]		: Card Model: 		0x74a1
  GPU[0]		: Card Vendor: 		Advanced Micro Devices, Inc. [AMD/ATI]
  GPU[0]		: Card SKU: 		M3000108
  GPU[0]		: Subsystem ID: 	0x74a1
  GPU[0]		: Device Rev: 		0x00
  GPU[0]		: Node ID: 		3
  GPU[0]		: GUID: 		41632
  GPU[0]		: GFX Version: 		gfx942
  ================================== End of ROCm SMI Log ===================================
  PyTorch  : 2.8.0+gitb2fb688
  HIP      : 7.0.51831-a3e329ad8
  GPU avail: True
  GPU 0    :   |  206.1 GB
  Disk free: 204 GB
  ✅ Enough disk for LoRA adapter training

  Model    : ./models/Qwen3-Coder-Next
  Adapter  : ./vuln_qwen_adapter
  4-bit    : False
  Max seq  : 2048


# 📥 Phase 3: Base Model Verification & Local Download
**Objective:** Ensure the 160GB base model weights are present on local disk.
* Checks for the local existence of the `Qwen3-Coder-Next` model directory to prevent redundant downloads.
* Configures `hf_transfer` to optimize download bandwidth from the Hugging Face Hub if weights are missing.
* Downloads the 80-billion parameter Mixture-of-Experts (MoE) model weights in native bfloat16 precision.
* Outputs the local disk utilization of the verified model directory.

In [3]:
# ---- Cell 3: Download Qwen3-Coder-Next weights locally ----

# Run once — skip if ./models/Qwen3-Coder-Next already exists

import os

from huggingface_hub import snapshot_download

os.makedirs("./models", exist_ok=True)

if not os.path.exists(MODEL_PATH) or not os.listdir(MODEL_PATH):

    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

    os.system("pip install -q hf_transfer")

    print("📥 Downloading Qwen3-Coder-Next (~160GB BF16)...")

    print("   This runs in the background — takes 20-40 min on fast NIC")

    snapshot_download(

        repo_id   = "Qwen/Qwen3-Coder-Next",

        local_dir = MODEL_PATH,

        local_dir_use_symlinks = False,

    )

    print("✅ Download complete")

else:

    print(f"✅ Model already on disk: {MODEL_PATH}")

    os.system(f"du -sh {MODEL_PATH}")

✅ Model already on disk: ./models/Qwen3-Coder-Next
149G	./models/Qwen3-Coder-Next


# 🗄️ Phase 4: Raw Dataset Ingestion & Schema Inspection
**Objective:** Download and preview the foundational vulnerability datasets.
* Ingests the `CVEfixes` and `All-CVE-Records` training datasets directly from Hugging Face.
* Caches the 531MB `JavaVFC Extended` JSONL dataset from Zenodo using a local system call.
* Loads all three datasets into memory and inspects row quantities and column definitions.
* Prints a truncated sample row from each dataset to verify the structural format of the text.

In [4]:
# ---- Cell 4: Download & Inspect All Datasets ----
from datasets import load_dataset
import subprocess, os

print("📥 Loading CVEfixes...")
ds_fixes = load_dataset("hitoshura25/cvefixes", split="train")

print("📥 Loading All-CVE-Records...")
ds_cve = load_dataset("AlicanKiraz0/All-CVE-Records-Training-Dataset", split="train")

# ── JavaVFC: download from Zenodo ──
os.makedirs("./data", exist_ok=True)
javavfc_file = "./data/javavfc_extended.jsonl"

if not os.path.exists(javavfc_file):
    print("📥 Downloading JavaVFC Extended (531 MB)...")
    subprocess.run([
        "wget", "-q",    # removed --show-progress
        "https://zenodo.org/records/13731781/files/javavfc_extended.jsonl",
        "-O", javavfc_file
    ], capture_output=True)
    print("✅ JavaVFC downloaded")
else:
    print(f"✅ JavaVFC already on disk: {javavfc_file}")

print("📥 Loading JavaVFC...")
ds_javavfc = load_dataset("json", data_files=javavfc_file, split="train")

# ── Inspect all three ──
for name, ds in [("CVEfixes", ds_fixes), ("All-CVE-Records", ds_cve), ("JavaVFC Extended", ds_javavfc)]:
    print(f"\n{'='*50}")
    print(f"🔍 {name}")
    print(f"{'='*50}")
    print(f"  Rows   : {len(ds)}")
    print(f"  Columns: {ds.column_names}")
    print(f"\nSample row [0]:")
    for k, v in ds[0].items():
        s = str(v)
        print(f"  {k}: {s[:120] + '...' if len(s) > 120 else s}")

📥 Loading CVEfixes...
📥 Loading All-CVE-Records...
✅ JavaVFC already on disk: ./data/javavfc_extended.jsonl
📥 Loading JavaVFC...

🔍 CVEfixes
  Rows   : 12987
  Columns: ['cve_id', 'hash', 'repo_url', 'cve_description', 'cvss2_base_score', 'cvss3_base_score', 'published_date', 'severity', 'cwe_id', 'cwe_name', 'cwe_description', 'commit_message', 'commit_date', 'version_tag', 'repo_total_files', 'repo_total_commits', 'file_paths', 'language', 'diff_stats', 'diff_with_context', 'vulnerable_code', 'fixed_code', 'security_keywords']

Sample row [0]:
  cve_id: CVE-2024-24746
  hash: d42a0ebe6632bd0c318560e4293a522634f60594
  repo_url: https://github.com/apache/mynewt-nimble
  cve_description: [{'lang': 'en', 'value': "Loop with Unreachable Exit Condition ('Infinite Loop') vulnerability in Apache NimBLE.\xa0\n\n...
  cvss2_base_score: None
  cvss3_base_score: None
  published_date: 2024-04-06T12:15Z
  severity: nan
  cwe_id: CWE-835
  cwe_name: Loop with Unreachable Exit Condition ('Infinite

# ⚙️ Phase 5: ChatML Standardization, Language Filtering & Dataset Merging
**Objective:** Clean, format, and merge raw data into a unified, balanced training split.
* Filters the `CVEfixes` and `All-CVE-Records` datasets to isolate Java-specific context and code strings.
* Converts the raw entries into a standardized `ChatML` schema with explicit system, user, and assistant roles.
* Formats `JavaVFC Extended` to position git diffs as the user prompt and commit messages as the target assistant response.
* Concatenates the three sources and caps the final volume at 5,000 samples to remain within the training time allocation.
* Executes a 90/10 train-test split and displays the final distribution count across all dataset sources.

In [5]:
# ---- Cell 5: Format, Filter Java, Merge Datasets ----
from datasets import concatenate_datasets
import re
from collections import Counter

# ── CVEfixes: real vulnerable→fixed Java code diffs ──
print("⚙️  Formatting CVEfixes...")

def format_cvefixes(ex):
    lang       = str(ex.get('language') or '').lower().strip()
    vuln_code  = (ex.get('vulnerable_code') or '').strip()
    fixed_code = (ex.get('fixed_code') or '').strip()

    if lang != 'java' or not vuln_code or not fixed_code:
        return {"text": None, "source": None}
    if len(vuln_code) < 30 or len(fixed_code) < 30:
        return {"text": None, "source": None}

    cve_raw = ex.get('cve_description')
    cve_desc = "Security vulnerability in Java code."
    if isinstance(cve_raw, list) and cve_raw:
        cve_desc = cve_raw[0].get('value', cve_desc)
    elif isinstance(cve_raw, dict):
        cve_desc = cve_raw.get('value', cve_desc)

    cve_id = ex.get('cve_id') or 'Unknown CVE'
    cwe    = ex.get('cwe_id') or ex.get('cwe_name') or 'CWE-unknown'

    vuln_code  = vuln_code[:3000]
    fixed_code = fixed_code[:3000]

    return {
        "text": (
            "<|im_start|>system\n"
            "You are an expert Java security engineer. Identify and fix security vulnerabilities precisely.<|im_end|>\n"
            "<|im_start|>user\n"
            f"Fix this vulnerable Java code.\n\n"
            f"CVE ID: {cve_id}\n"
            f"Context: {cve_desc}\n"
            f"CWE: {cwe}\n\n"
            f"Vulnerable Code:\n```java\n{vuln_code}\n```<|im_end|>\n"
            "<|im_start|>assistant\n"
            f"Fixed Code:\n```java\n{fixed_code}\n```<|im_end|>"
        ),
        "source": "cvefixes_diffs"
    }

java_fixes = ds_fixes.map(format_cvefixes)
java_fixes = java_fixes.filter(lambda x: x['text'] is not None)
print(f"  └─ {len(java_fixes)} Java patch pairs")

# ── All-CVE-Records: Java vulnerability knowledge ──
print("\n⚙️  Formatting All-CVE-Records...")

JAVA_RE = re.compile(
    r'\b(java|jvm|spring|tomcat|struts|jackson|maven|hibernate|log4j|javax|servlet)\b',
    re.IGNORECASE
)

def format_cve_records(ex):
    user = str(ex.get('User') or '').strip()
    asst = str(ex.get('Assistant') or '').strip()

    if not user or not asst or len(asst) < 100:
        return {"text": None, "source": None}
    if not JAVA_RE.search(user) and not JAVA_RE.search(asst):
        return {"text": None, "source": None}

    return {
        "text": (
            "<|im_start|>system\n"
            "You are an expert Java security engineer. Analyze and explain security vulnerabilities.<|im_end|>\n"
            f"<|im_start|>user\n{user}<|im_end|>\n"
            f"<|im_start|>assistant\n{asst}<|im_end|>"
        ),
        "source": "cve_knowledge"
    }

java_knowledge = ds_cve.map(format_cve_records)
java_knowledge = java_knowledge.filter(lambda x: x['text'] is not None)
print(f"  └─ {len(java_knowledge)} Java knowledge entries")

# ── JavaVFC Extended: real Java vulnerability-fixing commits ──
print("\n⚙️  Formatting JavaVFC Extended...")

def format_javavfc(ex):
    msg  = (ex.get('message') or '').strip()
    diff = (ex.get('diff_raw') or '').strip()

    if not diff or len(diff) < 50 or not msg:
        return {"text": None, "source": None}

    diff = diff[:4000]

    # 🔒 FIXED: The user provides the diff, the assistant must analyze it and output the message
    return {
        "text": (
            "<|im_start|>system\n"
            "You are an expert Java security engineer analyzing code changes and pull requests.<|im_end|>\n"
            "<|im_start|>user\n"
            f"Review the following Java code diff and explain the security vulnerability it resolves.\n\n"
            f"```diff\n{diff}\n```<|im_end|>\n"
            "<|im_start|>assistant\n"
            f"Based on the diff provided, here is the analysis of the security patch:\n\n"
            f"{msg}<|im_end|>"
        ),
        "source": "javavfc_diffs"
    }

java_vfc = ds_javavfc.map(format_javavfc)
java_vfc = java_vfc.filter(lambda x: x['text'] is not None)
print(f"  └─ {len(java_vfc)} JavaVFC entries")

# ── Merge all three ──
combined = concatenate_datasets([
    java_fixes.select_columns(['text', 'source']),
    java_knowledge.select_columns(['text', 'source']),
    java_vfc.select_columns(['text', 'source']),
])

# ── Cap if needed for time budget ──
MAX_TRAIN = 5000
if len(combined) > MAX_TRAIN:
    combined = combined.shuffle(seed=42).select(range(MAX_TRAIN))
    print(f"\n⚠️ Dataset capped to {MAX_TRAIN} samples to protect hackathon time budget.")

final_splits = combined.train_test_split(test_size=0.1, seed=42)

print("\n" + "="*50)
print("📊 FINAL DATASET DISTRIBUTION")
print("="*50)
print(f"  Train    : {len(final_splits['train'])}")
print(f"  Validate : {len(final_splits['test'])}")
print(f"  Total    : {len(combined)}")
print(f"  Mix      : {dict(Counter(combined['source']))}")

⚙️  Formatting CVEfixes...
  └─ 386 Java patch pairs

⚙️  Formatting All-CVE-Records...
  └─ 6155 Java knowledge entries

⚙️  Formatting JavaVFC Extended...
  └─ 16837 JavaVFC entries

⚠️ Dataset capped to 5000 samples to protect hackathon time budget.

📊 FINAL DATASET DISTRIBUTION
  Train    : 4500
  Validate : 500
  Total    : 5000
  Mix      : {'javavfc_diffs': 3625, 'cve_knowledge': 1290, 'cvefixes_diffs': 85}


# ⚖️ Phase 6: Baseline Model Evaluation (Pre-Training)
**Objective:** Capture the base model's default responses to Java security prompts for later comparison.
* Iterates through predefined CWE test cases and a JavaVFC diff analysis.
* Generates baseline outputs using Qwen's recommended generation parameters.
* Saves results to `before_outputs.json` to enable side-by-side evaluation post-training.
* Purges the base model from VRAM and clears the cache to ensure adequate memory for the fine-tuning phase.

In [5]:
# ---- Cell 6: Generate BEFORE outputs with BASE model ----
# ⚠️  Must run BEFORE fine-tuning. Saves outputs to disk. Deletes model after.

from unsloth import FastLanguageModel
from transformers import TextStreamer
import torch, json, gc

TEST_CASES = {
    "CWE-89 · SQL Injection": "Fix this vulnerable Java code:\n```java\npublic User login(String user, String pass) {\n    String sql = \"SELECT * FROM users WHERE name='\" + user\n               + \"' AND pass='\" + pass + \"'\";\n    return db.query(sql);\n}\n```",
    "CWE-78 · Command Injection": "Fix this vulnerable Java code:\n```java\npublic String ping(String host) throws Exception {\n    Process p = Runtime.getRuntime().exec(\"ping -c 1 \" + host);\n    return new String(p.getInputStream().readAllBytes());\n}\n```",
    "CWE-22 · Path Traversal": "Fix this vulnerable Java code:\n```java\npublic byte[] download(String filename) throws IOException {\n    return Files.readAllBytes(Paths.get(\"/var/www/uploads/\" + filename));\n}\n```",
    "CWE-611 · XXE Injection": "Fix this vulnerable Java code:\n```java\npublic Document parseXML(InputStream in) throws Exception {\n    DocumentBuilderFactory f = DocumentBuilderFactory.newInstance();\n    return f.newDocumentBuilder().parse(in);\n}\n```",
    
    # 🌟 NEW: The JavaVFC Proof-of-Concept Test
    "JavaVFC · Diff Analysis": "Review the following Java code diff and explain the security vulnerability it resolves.\n```diff\n- String query = \"SELECT * FROM items WHERE id = '\" + id + \"'\";\n- Statement stmt = connection.createStatement();\n- ResultSet rs = stmt.executeQuery(query);\n+ String query = \"SELECT * FROM items WHERE id = ?\";\n+ PreparedStatement pstmt = connection.prepareStatement(query);\n+ pstmt.setString(1, id);\n+ ResultSet rs = pstmt.executeQuery();\n```"
}

def generate_eval(model, tokenizer, user_prompt, max_new_tokens=2048, stream=True):
    prompt = (
        "<|im_start|>system\n"
        "You are an expert Java security engineer. Identify and fix security vulnerabilities precisely.<|im_end|>\n"
        "<|im_start|>user\n"
        f"{user_prompt}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    streamer = TextStreamer(tokenizer, skip_special_tokens=True) if stream else None
    
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            temperature        = 1.0,     # ✅ Restored to Qwen official recommendation
            top_p              = 0.95,    # ✅ Restored to Qwen official recommendation
            top_k              = 40,      # ✅ Restored to Qwen official recommendation
            min_p              = 0.01,
            repetition_penalty = 1.0,    # Gentle penalty to prevent looping
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id,
            streamer           = streamer,
        )
    return tokenizer.decode(
        out[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

print("Loading BASE model (bf16)...")
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name        = MODEL_PATH,
    max_seq_length    = MAX_SEQ_LEN,
    dtype             = None,
    load_in_4bit      = LOAD_IN_4BIT,
    trust_remote_code = True,
)
FastLanguageModel.for_inference(base_model)
print(f"✅ Base model loaded | VRAM: {torch.cuda.memory_allocated(0)/1e9:.1f} GB")

# Generate and save BEFORE outputs
before_outputs = {}
for name, prompt_text in TEST_CASES.items():
    print(f"\n{'='*50}\n⏳ [{name}] generating BEFORE output...\n{'='*50}")
    before_outputs[name] = {
        "prompt": prompt_text,
        "before": generate_eval(base_model, base_tokenizer, prompt_text)
    }
    print(f"\n✅ Done")

# Save to disk
with open("./before_outputs.json", "w") as f:
    json.dump(before_outputs, f, indent=2)
print("\n💾 BEFORE outputs saved → ./before_outputs.json")

# ── Free ALL memory before fine-tuning ──
del base_model, base_tokenizer
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"🧹 Memory cleared | VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0))/1e9:.1f} GB")

Loading BASE model (bf16)...
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.6.7: Fast Qwen3_Next patching. Transformers: 5.5.0. vLLM: 0.11.1rc2.dev161+g8a297115e.rocm700.
   \\   /|    AMD gfx942 GPU. Num GPUs = 1. Max memory: 191.984 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+gitb2fb688. ROCm Toolkit: 7.0.51831-a3e329ad8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/759 [00:00<?, ?it/s]

✅ Base model loaded | VRAM: 159.4 GB

⏳ [CWE-89 · SQL Injection] generating BEFORE output...
system
You are an expert Java security engineer. Identify and fix security vulnerabilities precisely.
user
Fix this vulnerable Java code:
```java
public User login(String user, String pass) {
    String sql = "SELECT * FROM users WHERE name='" + user
               + "' AND pass='" + pass + "'";
    return db.query(sql);
}
```
assistant


Both `max_new_tokens` (=2048) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The vulnerability in this code is **SQL Injection**. The code directly concatenates user input (`user` and `pass`) into the SQL query string, allowing attackers to manipulate the query by injecting malicious SQL code (e.g., using `' OR '1'='1` as input).

Here’s the **secure fix** using a **parameterized query** (also known as a prepared statement):

```java
public User login(String user, String pass) {
    String sql = "SELECT * FROM users WHERE name = ? AND pass = ?";
    return db.query(sql, user, pass); // Use parameterized query (prevents SQL injection)
}
```

### ✅ Key Security Improvements:
- **Parameterized Queries**: The `?` placeholders ensure that user inputs are treated strictly as data, not executable SQL code.
- **No String Concatenation**: Prevents attackers from breaking out of the intended query structure.

### 🔒 Additional Recommendations:
1. **Hash Passwords**: Never store or compare passwords in plaintext. Use a strong, salted hashing algorithm (e.g., bcrypt, PBKDF2

...

This code is vulnerable to **Path Traversal (Directory Traversal)**. An attacker could pass a filename like `../../../etc/passwd` to read arbitrary files on the server.

### ✅ Secure Fix:
```java
import java.io.IOException;
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.Paths;
import java.util.Objects;

public byte[] download(String filename) throws IOException {
    // Validate input
    Objects.requireNonNull(filename, "Filename cannot be null");
    
    // Normalize path and prevent directory traversal
    Path baseDir = Paths.get("/var/www/uploads");
    Path resolvedPath = baseDir.resolve(filename).normalize();
    
    // Ensure resolved path is still within the base directory
    if (!resolvedPath.startsWith(baseDir.toString())) {
        throw new SecurityException("Invalid file path: " + filename);
    }
    
    return Files.readAllBytes(resolvedPath);
}
```

### 🔍 Why This Works:
- **Normalization**: `normalize()` resolves `..` and `.` sequences

Both `max_new_tokens` (=2048) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


This code is vulnerable to **XXE (XML External Entity)** attacks because it uses the default `DocumentBuilderFactory` configuration, which allows the parsing of external entities and can lead to information disclosure, server-side request forgery (SSRF), or denial of service.

### 🔒 Secure Fix
Disable external entity resolution and related features by configuring the factory securely.

```java
import org.w3c.dom.Document;
import javax.xml.XMLConstants;
import javax.xml.parsers.DocumentBuilder;
import javax.xml.parsers.DocumentBuilderFactory;
import java.io.InputStream;

public Document parseXML(InputStream in) throws Exception {
    DocumentBuilderFactory factory = DocumentBuilderFactory.newInstance();
    
    // Secure configuration to prevent XXE
    factory.setFeature("http://apache.org/xml/features/disallow-doctype-decl", true);
    factory.setFeature("http://xml.org/sax/features/external-general-entities", false);
    factory.setFeature("http://xml.org/sax/features/external-param

Both `max_new_tokens` (=2048) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


This diff fixes a **SQL injection vulnerability**.

### Vulnerability Explanation:
The original code concatenates the `id` parameter directly into the SQL query string:
```java
String query = "SELECT * FROM items WHERE id = '" + id + "'";
```
If `id` is user-controlled input (e.g., from a form field or HTTP request parameter), an attacker could inject malicious SQL code. For example, if `id = "1' OR '1'='1"`, the resulting query becomes:
```sql
SELECT * FROM items WHERE id = '1' OR '1'='1'
```
This would return *all* rows from the table, potentially exposing sensitive data. Worse attacks could involve `UNION` queries, data extraction, or even `DROP TABLE` statements if the database permissions allow it.

### Fix Details:
The updated code uses a **parameterized query with `PreparedStatement`**:
```java
String query = "SELECT * FROM items WHERE id = ?";
PreparedStatement pstmt = connection.prepareStatement(query);
pstmt.setString(1, id);
```
- The `?` is a *placeholder* for a parameter, 

# 🧠 Phase 7: Model Initialization & Parameter-Efficient Tuning Setup
**Objective:** Load the base model and configure the LoRA adapter for targeted training.
* Sets `UNSLOTH_MOE_BACKEND` to `native` to ensure mathematical compatibility with the AMD ROCm environment.
* Loads the `Qwen3-Coder-Next` base weights into VRAM.
* Attaches a Rank-64 LoRA adapter specifically targeting standard attention layers (`q_proj`, `k_proj`, `v_proj`, `o_proj`).
* Validates the trainable parameter count to ensure memory utilization remains safely within hardware limits.

In [6]:
# ---- Cell 7: Load Qwen3-Coder-Next + Attach LoRA (VRAM SAFE) ----
from unsloth import FastLanguageModel
import torch
import os

# 🔒 CRITICAL: Forces Unsloth to use AMD-compatible MoE math
os.environ["UNSLOTH_MOE_BACKEND"] = "native"

print("Loading model for fine-tuning (bf16 LoRA)...")
qwen_model, qwen_tokenizer = FastLanguageModel.from_pretrained(
    model_name        = MODEL_PATH,
    max_seq_length    = MAX_SEQ_LEN,
    dtype             = None,
    load_in_4bit      = LOAD_IN_4BIT,
    trust_remote_code = True,
)

qwen_model = FastLanguageModel.get_peft_model(
    qwen_model,
    r                          = 64,
    lora_alpha                 = 128,
    lora_dropout               = 0,
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj"], # 🔒 CRITICAL: Only standard attention layers!
    bias                       = "none",
    use_gradient_checkpointing = "unsloth", 
    random_state               = 42,
)

qwen_model.print_trainable_parameters()
print(f"✅ LoRA attached | VRAM: {torch.cuda.memory_allocated(0)/1e9:.1f} GB")

Loading model for fine-tuning (bf16 LoRA)...
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.6.7: Fast Qwen3_Next patching. Transformers: 5.5.0. vLLM: 0.11.1rc2.dev161+g8a297115e.rocm700.
   \\   /|    AMD gfx942 GPU. Num GPUs = 1. Max memory: 191.984 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+gitb2fb688. ROCm Toolkit: 7.0.51831-a3e329ad8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/759 [00:00<?, ?it/s]

trainable params: 16,515,072 || all params: 79,690,906,368 || trainable%: 0.0207
✅ LoRA attached | VRAM: 159.4 GB


# ⚡ Phase 8: Supervised Fine-Tuning (SFT) Execution
**Objective:** Execute the fine-tuning process on the curated Java AppSec dataset.
* Configures the `SFTTrainer` for 1 complete epoch with an effective batch size of 16.
* Implements a cosine learning rate scheduler and bfloat16 precision for stable optimization.
* Explicitly sets `packing = False` to prevent cross-contamination between distinct vulnerability code samples.
* Initiates the training loop to align the model's generative outputs with the structured DevSecOps format.

In [7]:
# ---- Cell 8: ⚡⚡ FINE-TUNING STARTS HERE ⚡⚡ ----

from trl import SFTTrainer, SFTConfig

os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = SFTConfig(

    output_dir                  = OUTPUT_DIR,

    num_train_epochs            = 1,           # 1 solid epoch — plenty for domain alignment

    per_device_train_batch_size = 2,

    per_device_eval_batch_size  = 2,

    gradient_accumulation_steps = 8,           # effective batch = 16

    learning_rate               = 2e-4,

    weight_decay                = 0.01,

    lr_scheduler_type           = "cosine",

    warmup_ratio                = 0.03,

    bf16                        = True,

    logging_steps               = 10,

    logging_first_step          = True,

    eval_strategy               = "steps",

    eval_steps                  = 100,

    save_strategy               = "steps",

    save_steps                  = 200,

    save_total_limit            = 2,

    dataloader_num_workers      = 4,

    report_to                   = "none",

    max_seq_length              = MAX_SEQ_LEN,

    dataset_text_field          = "text",
    packing                     = False,

)

trainer = SFTTrainer(

    model             = qwen_model,

    args              = training_args,

    train_dataset     = final_splits["train"],

    eval_dataset      = final_splits["test"],

    processing_class  = qwen_tokenizer,

)

print("=" * 55)

print("⚡⚡ FINE-TUNING NOW RUNNING — Qwen3-Coder-Next ⚡⚡")

print("   Watch: loss should drop from ~2.x → ~0.x")

print("=" * 55)

trainer_stats = trainer.train()

print("\n" + "=" * 55)

print("✅ FINE-TUNING COMPLETE")

print(f"   Total steps    : {trainer_stats.global_step}")

print(f"   Training loss  : {trainer_stats.training_loss:.4f}")

print(f"   Total time     : {trainer_stats.metrics['train_runtime']/60:.1f} min")

print("=" * 55)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/4500 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/500 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
⚡⚡ FINE-TUNING NOW RUNNING — Qwen3-Coder-Next ⚡⚡
   Watch: loss should drop from ~2.x → ~0.x
Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
100,0.812959,0.810995
200,0.805779,0.801774
282,0.807660,0.800498


Unsloth: Restored added_tokens_decoder metadata in ./training_output/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./training_output/checkpoint-282/tokenizer_config.json.



✅ FINE-TUNING COMPLETE
   Total steps    : 282
   Training loss  : 0.8499
   Total time     : 46.3 min


# 💾 Phase 9: Adapter Export & Storage Optimization
**Objective:** Save the trained adapter weights and remove intermediate system data.
* Exports the fine-tuned LoRA adapter files and associated tokenizer to the local directory.
* Deletes heavy intermediate training checkpoints generated by Hugging Face to free system disk space.
* Outputs a final disk usage report to confirm storage limits are securely maintained.

In [8]:
# ---- Cell 9: Save LoRA Adapter Only (62GB free → merge impossible) ----
import os

os.makedirs(ADAPTER_DIR, exist_ok=True)

print("💾 Saving LoRA adapter...")
qwen_model.save_pretrained(ADAPTER_DIR)
qwen_tokenizer.save_pretrained(ADAPTER_DIR)
print(f"✅ Adapter saved → {ADAPTER_DIR}")
os.system(f"du -sh {ADAPTER_DIR}")

# ── Free up disk: delete training checkpoints (not needed anymore) ──
print("\n🧹 Removing training checkpoints to free disk...")
os.system(f"rm -rf {OUTPUT_DIR}/checkpoint-*")
print("✅ Checkpoints cleared")

print("\n📊 Current disk usage:")
os.system("df -h /workspace")
os.system("du -sh /workspace/models/* /workspace/vuln_qwen_adapter 2>/dev/null")

💾 Saving LoRA adapter...


Unsloth: Restored added_tokens_decoder metadata in ./vuln_qwen_adapter/tokenizer_config.json.


✅ Adapter saved → ./vuln_qwen_adapter
74M	./vuln_qwen_adapter

🧹 Removing training checkpoints to free disk...
✅ Checkpoints cleared

📊 Current disk usage:
Filesystem      Size  Used Avail Use% Mounted on
overlay         879G  622G  213G  75% /


256

# 📊 Phase 10: Post-Training Evaluation & Clean Baseline Comparison
**Objective:** Generate fine-tuned responses and compile a sanitized comparative analysis.
* Synchronizes with the `before_outputs.json` baseline if available in the local environment.
* Silences native Hugging Face token warnings for a cleaner evaluation output.
* Executes test cases without live streaming to enable regex-based suppression of raw ChatML tags (`system`, `user`, `assistant`).
* Formats the before-and-after results into a highly readable terminal matrix and saves the final data to `comparison_results.json`.

In [17]:
# ---- Phase 10: Post-Training Evaluation & Clean Baseline Comparison ----
import json, torch, os, re
from transformers import TextStreamer
import transformers

# ── Silence Hugging Face generation warnings ──
transformers.utils.logging.set_verbosity_error()

FastLanguageModel.for_inference(qwen_model)

TEST_CASES = {
    "CWE-89 · SQL Injection": "Fix this vulnerable Java code:\n```java\npublic User login(String user, String pass) {\n    String sql = \"SELECT * FROM users WHERE name='\" + user\n               + \"' AND pass='\" + pass + \"'\";\n    return db.query(sql);\n}\n```",
    "CWE-78 · Command Injection": "Fix this vulnerable Java code:\n```java\npublic String ping(String host) throws Exception {\n    Process p = Runtime.getRuntime().exec(\"ping -c 1 \" + host);\n    return new String(p.getInputStream().readAllBytes());\n}\n```",
    "CWE-22 · Path Traversal": "Fix this vulnerable Java code:\n```java\npublic byte[] download(String filename) throws IOException {\n    return Files.readAllBytes(Paths.get(\"/var/www/uploads/\" + filename));\n}\n```",
    "CWE-611 · XXE Injection": "Fix this vulnerable Java code:\n```java\npublic Document parseXML(InputStream in) throws Exception {\n    DocumentBuilderFactory f = DocumentBuilderFactory.newInstance();\n    return f.newDocumentBuilder().parse(in);\n}\n```",
    "JavaVFC · Diff Analysis": "Review the following Java code diff and explain the security vulnerability it resolves.\n```diff\n- String query = \"SELECT * FROM items WHERE id = '\" + id + \"'\";\n- Statement stmt = connection.createStatement();\n- ResultSet rs = stmt.executeQuery(query);\n+ String query = \"SELECT * FROM items WHERE id = ?\";\n+ PreparedStatement pstmt = connection.prepareStatement(query);\n+ pstmt.setString(1, id);\n+ ResultSet rs = pstmt.executeQuery();\n```"
}

before_outputs = {}
if os.path.exists("./before_outputs.json"):
    with open("./before_outputs.json") as f:
        before_outputs = json.load(f)
    print("✅ Successfully synchronized with Phase 6 baseline outputs.")
else:
    print("⚠️ Baseline binary missing; evaluating post-training performance only.")

def generate_eval_clean(model, tokenizer, user_prompt, max_new_tokens=2048):
    prompt = (
        "<|im_start|>system\n"
        "You are an expert Java security engineer. Identify and fix security vulnerabilities precisely.<|im_end|>\n"
        "<|im_start|>user\n"
        f"{user_prompt}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            temperature        = 1.0,
            top_p              = 0.95,
            top_k              = 40,
            min_p              = 0.01,
            repetition_penalty = 1.0,
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id,
        )
    
    # Extract only the newly generated tokens
    generated_text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    
    # Clean up any leftover ChatML tags or prompt echoes bleeding through
    clean_text = re.sub(r'^(system|user|assistant)\s*', '', generated_text, flags=re.IGNORECASE)
    clean_text = clean_text.replace("Fix this vulnerable Java code.", "").strip()
    return clean_text

comparison_results = {}

print("\n" + "="*60)
print("📊 EVALUATION MATRIX: BASELINE VS FINE-TUNED MODEL")
print("="*60)

for name, prompt_text in TEST_CASES.items():
    print(f"\n[+] IDENTIFIER: {name}")
    print("-" * 60)
    
    # Fetch baseline if available
    if name in before_outputs:
        base_raw = before_outputs[name]['before']
        # Trim baseline for cleaner notebook visualization if it is too long
        base_lines = base_raw.split('\n')
        base_preview = '\n'.join(base_lines[:15]) + "\n... [Truncated for Review]" if len(base_lines) > 15 else base_raw
        print(f"❌ PRE-TRAINING (Base Model Preview):\n{base_preview}")
        print("-" * 60)

    # Generate polished post-training output
    after = generate_eval_clean(qwen_model, qwen_tokenizer, prompt_text)
    print(f"✨ POST-TRAINING (Fine-Tuned Target Fix):\n{after}")
    print("=" * 60)

    comparison_results[name] = {
        "prompt": prompt_text,
        "before": before_outputs.get(name, {}).get("before", "N/A"),
        "after":  after,
    }

with open("./comparison_results.json", "w") as f:
    json.dump(comparison_results, f, indent=2)
print("\n💾 Analytical summary committed to disk → ./comparison_results.json")

✅ Successfully synchronized with Phase 6 baseline outputs.

📊 EVALUATION MATRIX: BASELINE VS FINE-TUNED MODEL

[+] IDENTIFIER: CWE-89 · SQL Injection
------------------------------------------------------------
❌ PRE-TRAINING (Base Model Preview):
The vulnerability in this code is **SQL Injection**. The code directly concatenates user input (`user` and `pass`) into the SQL query string, allowing attackers to manipulate the query by injecting malicious SQL code (e.g., using `' OR '1'='1` as input).

Here’s the **secure fix** using a **parameterized query** (also known as a prepared statement):

```java
public User login(String user, String pass) {
    String sql = "SELECT * FROM users WHERE name = ? AND pass = ?";
    return db.query(sql, user, pass); // Use parameterized query (prevents SQL injection)
}
```

### ✅ Key Security Improvements:
- **Parameterized Queries**: The `?` placeholders ensure that user inputs are treated strictly as data, not executable SQL code.
- **No String Conc

# 📋 Phase 11: Deep-Dive Vulnerability & Verification Report (Deliverable)
**Objective:** Demonstrate enterprise-grade DevSecOps capabilities via automated auditing.
* Shifts the model persona to a "Lead Java AppSec Reviewer" using a highly structured prompt.
* Analyzes a raw SQL injection vulnerability and generates a formal, multi-section security audit.
* Forces the model to generate a **Comparative Analysis** and **Fix Verification** to logically defend why its applied patch mathematically neutralizes the exploit.
* Streams the final report live and saves it to `sample_vuln_report.txt` as the primary hackathon deliverable.

In [20]:
# ---- Phase 11: Deep-Dive Vulnerability & Verification Report (Deliverable) ----
from transformers import TextStreamer
import torch
import transformers

# Silence Hugging Face logging noise
transformers.utils.logging.set_verbosity_error()

def generate_vuln_report(model, tokenizer, vuln_code):
    prompt = (
        "<|im_start|>system\n"
        "You are an expert Java security engineer. Analyze and explain security vulnerabilities.<|im_end|>\n"
        "<|im_start|>user\n"
        "Analyze this vulnerable Java code and produce a detailed, structured DevSecOps report with the following exact sections:\n"
        "1. Threat Classification (Vulnerability Type, CWE, CVSS Estimate)\n"
        "2. Attack Vector & Proof-of-Concept\n"
        "3. Secure Fixed Code\n"
        "4. Comparative Analysis (Explain exactly what changed between the vulnerable and fixed code)\n"
        "5. Fix Verification (Prove why the new code successfully neutralizes the exploit)\n"
        "6. Production Remediation Notes\n\n"
        f"```java\n{vuln_code}\n```<|im_end|>\n"
        "<|im_start|>assistant\n"
        "Here is the detailed DevSecOps security report:\n\n### 1. Threat Classification\n" # 🎯 Forces the model to start the list!
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens     = 2048,
            temperature        = 1.0,
            top_p              = 0.95,
            top_k              = 40,
            repetition_penalty = 1.0,     # Removed penalty to prevent early cutoffs
            pad_token_id       = tokenizer.eos_token_id,
            streamer           = streamer,
        )
    
    # Clean the output to ensure no raw prompt text bleeds into the saved file
    generated_text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    clean_text = generated_text.replace("Analyze this vulnerable Java code", "").strip()
    return clean_text

DEMO = """@RequestMapping("/search")
public String search(@RequestParam String query, Model model) {
    String sql = "SELECT * FROM products WHERE name LIKE '%" + query + "%'";
    List<Product> r = jdbcTemplate.query(sql, new ProductMapper());
    model.addAttribute("results", r);
    return "results";
}"""

print(f"\n{'='*60}")
print("📋 GENERATING DEEP-DIVE SECURITY REPORT...")
print(f"{'='*60}\n")

report = generate_vuln_report(qwen_model, qwen_tokenizer, DEMO)

with open("./sample_vuln_report.txt", "w") as f:
    f.write(f"VULNERABLE CODE:\n{DEMO}\n\nREPORT:\n{report}")
print("\n💾 Report saved → ./sample_vuln_report.txt")


📋 GENERATING DEEP-DIVE SECURITY REPORT...

- **Vulnerability Type**: SQL Injection
- **CWE**: CWE-89
- **CVSS Estimate**: 9.8 (Critical)

### 2. Attack Vector & Proof-of-Concept
An attacker could exploit this vulnerable code by crafting malicious input in the `query` parameter. For example:
```
/s/search?query=' OR '1'='1
```
This would result in the query:
```sql
SELECT * FROM products WHERE name LIKE '%' OR '1'='1%'
```
This query would return all records from the `products` table, bypassing the intended search functionality.

### 3. Secure Fixed Code
```java
@RequestMapping("/search")
public String search(@RequestParam String query, Model model) {
    String sql = "SELECT * FROM products WHERE name LIKE ?";
    List<Product> r = jdbcTemplate.query(sql, new Object[]{"%"+query+"%"}, new ProductMapper());
    model.addAttribute("results", r);
    return "results";
}
```

### 4. Comparative Analysis

**Vulnerable Code:**
```java
String sql = "SELECT * FROM products WHERE name LIKE '%" 

In [20]:
# ---- Phase 11: Deep-Dive Vulnerability & Verification Report (Deliverable) ----
from transformers import TextStreamer
import torch
import transformers

# Silence Hugging Face logging noise
transformers.utils.logging.set_verbosity_error()

def generate_vuln_report(model, tokenizer, vuln_code):
    prompt = (
        "<|im_start|>system\n"
        "You are an expert Java security engineer. Analyze and explain security vulnerabilities.<|im_end|>\n"
        "<|im_start|>user\n"
        "Analyze this vulnerable Java code and produce a detailed, structured DevSecOps report with the following exact sections:\n"
        "1. Threat Classification (Vulnerability Type, CWE, CVSS Estimate)\n"
        "2. Attack Vector & Proof-of-Concept\n"
        "3. Secure Fixed Code\n"
        "4. Comparative Analysis (Explain exactly what changed between the vulnerable and fixed code)\n"
        "5. Fix Verification (Prove why the new code successfully neutralizes the exploit)\n"
        "6. Production Remediation Notes\n\n"
        f"```java\n{vuln_code}\n```<|im_end|>\n"
        "<|im_start|>assistant\n"
        "Here is the detailed DevSecOps security report:\n\n### 1. Threat Classification\n" # 🎯 Forces the model to start the list!
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens     = 2048,
            temperature        = 1.0,
            top_p              = 0.95,
            top_k              = 40,
            repetition_penalty = 1.0,     # Removed penalty to prevent early cutoffs
            pad_token_id       = tokenizer.eos_token_id,
            streamer           = streamer,
        )
    
    # Clean the output to ensure no raw prompt text bleeds into the saved file
    generated_text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    clean_text = generated_text.replace("Analyze this vulnerable Java code", "").strip()
    return clean_text

DEMO = """@RequestMapping("/search")
public String search(@RequestParam String query, Model model) {
    String sql = "SELECT * FROM products WHERE name LIKE '%" + query + "%'";
    List<Product> r = jdbcTemplate.query(sql, new ProductMapper());
    model.addAttribute("results", r);
    return "results";
}"""

print(f"\n{'='*60}")
print("📋 GENERATING DEEP-DIVE SECURITY REPORT...")
print(f"{'='*60}\n")

report = generate_vuln_report(qwen_model, qwen_tokenizer, DEMO)

with open("./sample_vuln_report.txt", "w") as f:
    f.write(f"VULNERABLE CODE:\n{DEMO}\n\nREPORT:\n{report}")
print("\n💾 Report saved → ./sample_vuln_report.txt")


📋 GENERATING DEEP-DIVE SECURITY REPORT...

- **Vulnerability Type**: SQL Injection
- **CWE**: CWE-89
- **CVSS Estimate**: 9.8 (Critical)

### 2. Attack Vector & Proof-of-Concept
An attacker could exploit this vulnerable code by crafting malicious input in the `query` parameter. For example:
```
/s/search?query=' OR '1'='1
```
This would result in the query:
```sql
SELECT * FROM products WHERE name LIKE '%' OR '1'='1%'
```
This query would return all records from the `products` table, bypassing the intended search functionality.

### 3. Secure Fixed Code
```java
@RequestMapping("/search")
public String search(@RequestParam String query, Model model) {
    String sql = "SELECT * FROM products WHERE name LIKE ?";
    List<Product> r = jdbcTemplate.query(sql, new Object[]{"%"+query+"%"}, new ProductMapper());
    model.addAttribute("results", r);
    return "results";
}
```

### 4. Comparative Analysis

**Vulnerable Code:**
```java
String sql = "SELECT * FROM products WHERE name LIKE '%" 

# The End

In [ ]:
# ---- Phase 13: Package Solution for GitHub / Submission ----
import os
import zipfile
from IPython.display import FileLink, display

def package_solution(output_filename="FINETUNING_001_Submission.zip"):
    # 🔒 Explicitly defining targets ensures the 160GB base model is ignored!
    targets = [
        "Fine Tuned Qwen3-Coder-Next for Java Vulns.ipynb", # <-- Update this if your notebook name is different!
        "before_outputs.json",
        "comparison_results.json",
        "sample_vuln_report.txt",
        "vuln_qwen_adapter"                 # Your LoRA weights (Small, safe to download)
    ]

    print(f"⏳ Zipping project files into {output_filename}...")
    
    with zipfile.ZipFile(output_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for target in targets:
            if os.path.isfile(target):
                zipf.write(target)
                print(f"  + Added file: {target}")
            elif os.path.isdir(target):
                for root, dirs, files in os.walk(target):
                    for file in files:
                        file_path = os.path.join(root, file)
                        # Maintain folder structure inside the zip
                        zipf.write(file_path, os.path.relpath(file_path, '.'))
                print(f"  + Added directory: {target}/")
            else:
                print(f"  ⚠️ Warning: '{target}' not found. Skipping.")

    size_mb = os.path.getsize(output_filename) / (1024 * 1024)
    print(f"\n✅ Packaging complete! Total size: {size_mb:.2f} MB")
    
    return output_filename

# Run the packager and create a clickable download link
zip_file = package_solution()
print("\n👇 CLICK THE LINK BELOW TO DOWNLOAD TO YOUR LOCAL MACHINE 👇")
display(FileLink(zip_file))